# Cell 1
### Written for per page structure

In [17]:
from lxml import etree
from pathlib import Path
import re

# Root data directory containing all issue folders
data_dir = Path("/Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/xml_data")

# Find all issue folders matching the naming pattern
issue_folders = sorted(data_dir.glob("heresies_*_pagexml"))

print(f"Found {len(issue_folders)} issue folders:")
for folder in issue_folders:
    # Extract issue number from folder name
    issue_num = re.search(r"heresies_(\d+)_pagexml", folder.name).group(1)
    page_files = sorted(folder.glob("page_*.xml"), key=lambda f: int(re.search(r"page_(\d+)", f.stem).group(1)))
    print(f"  Issue {issue_num}: {len(page_files)} pages")

Found 1 issue folders:
  Issue 01: 116 pages


# Cell 2

In [18]:
def parse_custom(custom_string):
    type_match = re.search(r"type:([^;}\s]+)", custom_string)
    structure_type = type_match.group(1) if type_match else None
    
    order_match = re.search(r"readingOrder\s*\{index:(\d+);?\}", custom_string)
    reading_order = int(order_match.group(1)) if order_match else None
    
    return {
        "type": structure_type,
        "reading_order": reading_order
    }


def extract_regions_from_page(page_file, ns, issue_num, page_num):
    """
    Parses a single page XML file and returns a list of region dictionaries,
    sorted by reading order index within that page.
    """
    
    tree = etree.parse(str(page_file))
    root = tree.getroot()
    
    raw_regions = []
    
    for region in root.findall(".//page:TextRegion", ns):
        custom_string = region.get("custom", "")
        parsed = parse_custom(custom_string)
        
        lines = []
        for line in region.findall(".//page:TextLine", ns):
            unicode_el = line.find(".//page:Unicode", ns)
            if unicode_el is not None and unicode_el.text:
                lines.append(unicode_el.text)
        
        raw_regions.append({
            "issue": issue_num,
            "page": page_num,
            "id": region.get("id"),
            "type": parsed["type"],
            "reading_order": parsed["reading_order"],
            "lines": lines,
            "raw_text": " ".join(lines)
        })
    
    raw_regions.sort(key=lambda r: (r["reading_order"] is None, r["reading_order"]))
    
    return raw_regions

## Cell 2B - Testing on One Issue

In [19]:
ns = {"page": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

# Test on just the first issue folder
test_folder = issue_folders[0]
issue_num = re.search(r"heresies_(\d+)_pagexml", test_folder.name).group(1)
page_files = sorted(test_folder.glob("page_*.xml"), key=lambda f: int(re.search(r"page_(\d+)", f.stem).group(1)))

# Extract all regions across all pages of this issue
all_regions_issue_01 = []

for page_file in page_files:
    page_num = int(re.search(r"page_(\d+)", page_file.stem).group(1))
    regions = extract_regions_from_page(page_file, ns, issue_num, page_num)
    all_regions_issue_01.extend(regions)

print(f"Total regions extracted from issue {issue_num}: {len(all_regions_issue_01)}")

# Inspect the first 10 in order
for r in all_regions_issue_01[:10]:
    print(f"Page {r['page']} | Order {r['reading_order']} | Type: {r['type']}")
    print(f"Preview: {r['raw_text'][:120]}")
    print()

Total regions extracted from issue 01: 1029
Page 1 | Order 0 | Type: issue_front_page
Preview: HERESIES a Feminist Publication on art and Politics

Page 2 | Order 0 | Type: paragraph
Preview: HERESIES is an idea-oriented journal devoted to the examination of art and politics from a feminist perspective. We beli

Page 2 | Order 1 | Type: paragraph
Preview: HERESIES is structured as a collective of fem- inists, some of whom are also socialists, Marx- ists, lesbian feminists o

Page 2 | Order 2 | Type: paragraph
Preview: issues.) Possibly satellite pamphlets and broad- sides will be produced continuing the discus- .

Page 2 | Order 3 | Type: paragraph
Preview: As women, we are aware that historically the connections between our lives, our arts and our ideas have been suppressed.

Page 2 | Order 4 | Type: paragraph
Preview: THE COLLECTIVE: Patsy Beckert, Joan Brader- man, Mary Beth Edelson, Harmony Hammond, Elizabeth Hess, Joyce Kozloff, Arle

Page 2 | Order 5 | Type: paragraph
Preview: HE

# Cell 3 - Region Filtering

In [45]:
# Define which region types to keep for each hypothesis
# Adjust these sets based on your theoretical decisions

FILTER_SETS = {
    "h1": {"paragraph", "heading"},
    "h2": {"paragraph", "reference_list"},
    "h3": {"paragraph", "heading", "author_creator"}
}

# Also define what to always exclude regardless of hypothesis
EXCLUDE_TYPES = {"page-number", "page_number", "issue_front_page", "caption"}


def filter_regions(regions, hypothesis):
    """
    Filters a list of region dictionaries by the types relevant 
    to a given hypothesis. Excludes types in EXCLUDE_TYPES.
    
    hypothesis: one of "h1", "h2", "h3"
    """
    
    allowed_types = FILTER_SETS[hypothesis]
    
    return [
        r for r in regions
        if r["type"] in allowed_types
        and r["type"] not in EXCLUDE_TYPES
    ]

## Cell 3B - Testing the Filter

In [21]:
# test the filter
for hypothesis in ["h1", "h2", "h3"]:
    filtered = filter_regions(all_regions_issue_01, hypothesis)
    print(f"{hypothesis}: {len(filtered)} regions retained from {len(all_regions_issue_01)} total")
    
    # Show which types survived
    surviving_types = set(r["type"] for r in filtered)
    print(f"  Types kept: {surviving_types}")
    print()

h1: 824 regions retained from 1029 total
  Types kept: {'paragraph', 'heading'}

h2: 772 regions retained from 1029 total
  Types kept: {'paragraph', 'reference_list'}

h3: 852 regions retained from 1029 total
  Types kept: {'paragraph', 'author_creator', 'heading'}



# Cell 4 - Cleaning

## Cell 4A - Dehyphenation

In [22]:
def dehyphenate_lines(lines):
    """
    Takes a list of strings (one per TextLine) and joins them,
    fixing hyphenated line breaks in the process.
    
    Logic: if a line ends with a hyphen, the last word is split
    across lines — remove the hyphen and join directly to the 
    next line without a space.
    If a line does NOT end with a hyphen, join with a space.
    """
    
    result = ""
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        if not line:
            continue
        
        if result == "":
            result = line
        elif result.endswith("-"):
            # Remove hyphen and join directly — word continues on next line
            result = result[:-1] + line
        else:
            # Normal line break — join with space
            result = result + " " + line
    
    return result

## Cell 4B - Text Normalization

In [23]:
import re

def normalize_text(text):
    """
    Applies post-dehyphenation cleaning to a joined text string.
    """
    
    # Collapse multiple whitespace characters into a single space
    text = re.sub(r"\s+", " ", text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    # Optionally: normalize unicode characters
    # e.g. curly quotes → straight quotes, ligatures → standard letters
    # import unicodedata
    # text = unicodedata.normalize("NFC", text)
    
    return text

## Cell 4C - Combine into one cleaning function

In [24]:
def clean_region(region):
    """
    Takes a region dictionary and returns it with a new 
    'clean_text' field added, preserving all other fields.
    """
    
    dehyphenated = dehyphenate_lines(region["lines"])
    cleaned = normalize_text(dehyphenated)
    
    # Add clean_text without overwriting raw_text
    # You want to keep raw_text for debugging and comparison
    return {**region, "clean_text": cleaned}


def clean_regions(regions):
    """Applies clean_region to a list of regions."""
    return [clean_region(r) for r in regions]

## Cell 4D - Test the cleaning

In [25]:
# Filter for H1 and then clean
filtered_h1 = filter_regions(all_regions_issue_01, "h1")
cleaned_h1 = clean_regions(filtered_h1)

# Spot check: find regions that had hyphens and verify they were fixed
print("Checking dehyphenation — showing regions that contained hyphens:\n")

for r in cleaned_h1[:50]:
    if "-" in r["raw_text"]:
        print(f"Page {r['page']} | Order {r['reading_order']}")
        print(f"  RAW:   {r['raw_text'][:150]}")
        print(f"  CLEAN: {r['clean_text'][:150]}")
        print()

Checking dehyphenation — showing regions that contained hyphens:

Page 2 | Order 0
  RAW:   HERESIES is an idea-oriented journal devoted to the examination of art and politics from a feminist perspective. We believe that what is commonly call
  CLEAN: HERESIES is an idea-oriented journal devoted to the examination of art and politics from a feminist perspective. We believe that what is commonly call

Page 2 | Order 1
  RAW:   HERESIES is structured as a collective of fem- inists, some of whom are also socialists, Marx- ists, lesbian feminists or anarchists; our fields inclu
  CLEAN: HERESIES is structured as a collective of feminists, some of whom are also socialists, Marxists, lesbian feminists or anarchists; our fields include p

Page 2 | Order 2
  RAW:   issues.) Possibly satellite pamphlets and broad- sides will be produced continuing the discus- .
  CLEAN: issues.) Possibly satellite pamphlets and broadsides will be produced continuing the discus.

Page 2 | Order 3
  RAW:   As wom

# Cell 5 - Merge Pages into Issue-Level Text and Build Corpus

## Cell 5A - Merge Regions into Issue-Level Text

In [26]:
def merge_issue_text(cleaned_regions):
    """
    Concatenates clean_text from all regions in reading order
    (which is already guaranteed by your page/order sorting).
    Joins regions with a double newline to preserve 
    rough paragraph boundaries.
    """
    
    return "\n\n".join(
        r["clean_text"] for r in cleaned_regions 
        if r["clean_text"]  # skip any empty strings
    )

## Cell 5B - Build a structured Record per Issue

In [27]:
# You'll need a manual mapping of issue number to publication year
# Build this from your knowledge of the magazine's publication history

ISSUE_YEAR_MAP = {
    "01": 1977,
    "02": 1977,
    # ... continue for all issues
}


def build_issue_record(issue_num, cleaned_regions, hypothesis):
    """
    Builds a structured dictionary representing one issue
    for a given hypothesis.
    """
    
    merged_text = merge_issue_text(cleaned_regions)
    
    return {
        "issue": issue_num,
        "year": ISSUE_YEAR_MAP.get(issue_num, None),
        "hypothesis": hypothesis,
        "num_regions": len(cleaned_regions),
        "text": merged_text
    }

## Cell 5C - Run the full Pipeline for Issue 1

In [28]:
corpus = []

for hypothesis in ["h1", "h2", "h3"]:
    filtered = filter_regions(all_regions_issue_01, hypothesis)
    cleaned = clean_regions(filtered)
    record = build_issue_record("01", cleaned, hypothesis)
    corpus.append(record)
    
    print(f"Issue 01 | {hypothesis} | {record['num_regions']} regions | {len(record['text'])} characters")

Issue 01 | h1 | 824 regions | 345556 characters
Issue 01 | h2 | 772 regions | 360425 characters
Issue 01 | h3 | 852 regions | 346025 characters


## Cell 5D - Inspect the Output

In [29]:
# Preview the merged text for H1
h1_record = next(r for r in corpus if r["hypothesis"] == "h1")

print(f"Issue: {h1_record['issue']} | Year: {h1_record['year']}")
print(f"Total characters: {len(h1_record['text'])}")
print(f"\nFirst 500 characters:\n")
print(h1_record["text"][:500])

Issue: 01 | Year: 1977
Total characters: 345556

First 500 characters:

HERESIES is an idea-oriented journal devoted to the examination of art and politics from a feminist perspective. We believe that what is commonly called art can have a political impact, and that in the making of art and of all cultural artifacts our identities as women play a distinct role. We hope that HERESIES will stimulate dialogue around radical political and esthetic theory, encourage the writing of the history of femina sapiens, and generate new creative energies among women. It will be a


# Cell 6 - Scaling the Pipeline to all issues

## Cell 6A - Full Pipeline Function

In [30]:
def process_issue(issue_folder, ns, hypothesis):
    """
    Runs the complete pipeline for one issue folder and one hypothesis.
    Returns a structured record dictionary.
    """
    
    # Extract issue number from folder name
    issue_num = re.search(r"heresies_(\d+)_pagexml", issue_folder.name).group(1)
    
    # Get page files sorted numerically
    page_files = sorted(
        issue_folder.glob("page_*.xml"),
        key=lambda f: int(re.search(r"page_(\d+)", f.stem).group(1))
    )
    
    # Extract all regions across all pages
    all_regions = []
    for page_file in page_files:
        page_num = int(re.search(r"page_(\d+)", page_file.stem).group(1))
        regions = extract_regions_from_page(page_file, ns, issue_num, page_num)
        all_regions.extend(regions)
    
    # Filter, clean, merge, build record
    filtered = filter_regions(all_regions, hypothesis)
    cleaned = clean_regions(filtered)
    record = build_issue_record(issue_num, cleaned, hypothesis)
    
    return record

## Cell 6B - Run Across All Issues and Hypotheses

In [31]:
full_corpus = []
failed = []

for issue_folder in issue_folders:
    for hypothesis in ["h1", "h2", "h3"]:
        try:
            record = process_issue(issue_folder, ns, hypothesis)
            full_corpus.append(record)
            print(f"✓ Issue {record['issue']} | {hypothesis} | {record['num_regions']} regions | year: {record['year']}")
        except Exception as e:
            failed.append((issue_folder.name, hypothesis, str(e)))
            print(f"✗ Failed: {issue_folder.name} | {hypothesis} | {e}")

print(f"\nDone. {len(full_corpus)} records built. {len(failed)} failures.")

✓ Issue 01 | h1 | 824 regions | year: 1977
✓ Issue 01 | h2 | 772 regions | year: 1977
✓ Issue 01 | h3 | 852 regions | year: 1977

Done. 3 records built. 0 failures.


## Cell 6C - Review any failures

In [32]:
if failed:
    print("Failed records:")
    for issue, hypothesis, error in failed:
        print(f"  {issue} | {hypothesis} | {error}")
else:
    print("No failures.")

No failures.


## Cell 6D - Sanity Check the full corpus

In [34]:
import pandas as pd

# Convert to dataframe just for inspection — not your final export yet
df_check = pd.DataFrame([
    {
        "issue": r["issue"],
        "year": r["year"],
        "hypothesis": r["hypothesis"],
        "num_regions": r["num_regions"],
        "num_characters": len(r["text"])
    }
    for r in full_corpus
])

# Pivot so you can compare issues side by side
print(df_check.pivot(index="issue", columns="hypothesis", values="num_regions"))
print()
print(df_check.pivot(index="issue", columns="hypothesis", values="num_characters"))

hypothesis   h1   h2   h3
issue                    
01          824  772  852

hypothesis      h1      h2      h3
issue                             
01          345556  360425  346025


# Cell 7 - Export

## Cell 7A - Setup Output Directory

In [40]:
output_dir = Path("/Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/processed")

# Create subdirectories for each hypothesis
for h in ["h1", "h2", "h3"]:
    (output_dir / h).mkdir(parents=True, exist_ok=True)

print("Output directories created.")

Output directories created.


## Cell 7B - Export CSV for All Hypotheses

In [41]:
import pandas as pd

df_full = pd.DataFrame([
    {
        "issue": r["issue"],
        "year": r["year"],
        "num_regions": r["num_regions"],
        "text": r["text"]
    }
    for r in full_corpus
    if r["hypothesis"] == "h1"  # change to h2 or h3 as needed
])

# Export — repeat for h2 and h3 by changing the filter and filename
df_full.to_csv(output_dir / "h1" / "corpus_h1.csv", index=False, encoding="utf-8-sig")
print("Exported corpus_h1.csv")

Exported corpus_h1.csv


## Cell 7C - Export Plain Text Files per Issue (for NLP tools)

In [42]:
# One .txt file per issue, for each hypothesis
for r in full_corpus:
    h = r["hypothesis"]
    filename = f"issue_{r['issue']}_{r['year']}.txt"
    filepath = output_dir / h / filename
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(r["text"])

print(f"Exported {len(full_corpus)} plain text files.")

Exported 3 plain text files.


## Cell 7D - Export JSON (for structured/article-level work)

In [43]:
import json

# Export full corpus as JSON — useful if you later add article-level segmentation
with open(output_dir / "full_corpus.json", "w", encoding="utf-8") as f:
    json.dump(full_corpus, f, ensure_ascii=False, indent=2)

print("Exported full_corpus.json")

Exported full_corpus.json


## Cell 7E - Final Verification

In [44]:
# Confirm all expected files exist
for h in ["h1", "h2", "h3"]:
    files = list((output_dir / h).glob("*"))
    print(f"{h}: {len(files)} files exported")
    for f in sorted(files):
        print(f"  {f.name}")

h1: 2 files exported
  corpus_h1.csv
  issue_01_1977.txt
h2: 1 files exported
  issue_01_1977.txt
h3: 1 files exported
  issue_01_1977.txt
